# Generation and Grounding in RAG

> **How do we make the LLM answer from retrieved evidence rather than simply generate a plausible answer?**

Our RAG pipeline now looks like:

```text
Documents
    ↓
Ingestion
    ↓
Chunking
    ↓
Embeddings
    ↓
Retrieval
    ↓
Hybrid Retrieval
    ↓
Reranking
    ↓
Context Engineering
```

We have deliberately built the pipeline so that the LLM receives useful evidence.

Now we reach the generation stage.

The central problem is:

> **How do we make the generated answer stay grounded in the evidence we retrieved?**

A RAG system can retrieve the correct information and still produce a poor answer.

The model may:

- Ignore the retrieved evidence
- Add unsupported facts
- Mix information from different sources
- Answer when the evidence is insufficient
- Misrepresent what a source says
- Produce an answer without useful citations

This notebook focuses on **grounded generation**: designing the generation stage so that answers are based on the retrieved context and uncertainty is handled explicitly.

## What We'll Learn

By the end of this tutorial, you'll understand:

- What grounded generation means
- Why retrieval quality alone is not enough
- How retrieved context is passed to an LLM
- How prompts can constrain generation
- Grounded answers vs. plausible answers
- What to do when evidence is insufficient
- Abstention
- Source attribution and citations
- Faithfulness and groundedness
- Structured generation
- How to inspect generated answers for unsupported claims
- How generation fits into the complete RAG pipeline

# 1. Retrieval Does Not Automatically Ground the Answer

Consider this pipeline:

```text
Query
  ↓
Retriever
  ↓
Correct evidence
  ↓
LLM
  ↓
Answer
```

It is tempting to assume:

> If the retriever found the correct evidence, the answer must be correct.

That assumption is wrong.

The LLM is still a generative model.

It can use information from the context, but it can also produce information that was not supported by the context.

Therefore:

```text
Good retrieval
      ≠
Guaranteed grounded generation
```

Grounding is its own engineering problem.

# 2. What Does "Grounded" Mean?

A grounded answer is an answer whose claims are supported by the available evidence.

For example, suppose the context says:

```text
Customers can request a refund within 30 days of purchase.
```

A grounded answer could be:

```text
Customers can request a refund within 30 days of purchase.
```

An unsupported answer might be:

```text
Customers can request a refund within 30 days, and refunds are always approved.
```

The second statement introduces a claim that the provided evidence does not establish.

The key distinction is:

```text
Evidence
   ↓
Supported claim
```

versus:

```text
Evidence
   ↓
Supported claim
   +
Unsupported inference
```

A strong RAG system should minimize unsupported claims.

# 3. Create Example Evidence

We'll use a small context containing a few policy documents.

In [1]:
context = """
[Source 1]
Document: refund_policy.md
Content: Customers can request a refund within 30 days of purchase.

[Source 2]
Document: refund_process.md
Content: Approved refunds are normally processed within 7 business days.

[Source 3]
Document: refund_method.md
Content: Refunds are returned to the original payment method used for the purchase.
"""

print(context)


[Source 1]
Document: refund_policy.md
Content: Customers can request a refund within 30 days of purchase.

[Source 2]
Document: refund_process.md
Content: Approved refunds are normally processed within 7 business days.

[Source 3]
Document: refund_method.md
Content: Refunds are returned to the original payment method used for the purchase.



Let's define two questions.

One can be answered directly from the evidence.

The other asks for information that is not present.

In [2]:
supported_question = "How long do customers have to request a refund?"

unsupported_question = "Are all refund requests automatically approved?"

print(supported_question)
print(unsupported_question)

How long do customers have to request a refund?
Are all refund requests automatically approved?


The first question has direct evidence.

The second does not.

This distinction is essential for grounded generation.

# 4. A Simple Grounded Generation Prompt

A basic grounded-generation prompt can explicitly tell the model to use the supplied context.

For example:

```text
Answer the question using only the provided context.

If the context does not contain enough information,
say that the information is not available.

Do not invent facts.
```

The prompt is not a guarantee, but it establishes the generation policy.

The general structure is:

```text
System instructions
        +
Retrieved context
        +
User question
        ↓
      LLM
        ↓
Grounded answer
```

# 5. Build the Prompt

Let's construct the prompt ourselves before connecting it to an LLM.

In [3]:
def build_grounded_prompt(question, context):
    return f"""
You are a question-answering assistant.

Answer the user's question using only the provided context.

Rules:
1. Use only information supported by the context.
2. Do not invent facts.
3. If the context does not contain enough information,
   clearly say that the information is not available.
4. When possible, identify the source supporting the answer.

Context:
{context}

Question:
{question}

Answer:
""".strip()

prompt = build_grounded_prompt(
    supported_question,
    context
)

print(prompt)

You are a question-answering assistant.

Answer the user's question using only the provided context.

Rules:
1. Use only information supported by the context.
2. Do not invent facts.
3. If the context does not contain enough information,
   clearly say that the information is not available.
4. When possible, identify the source supporting the answer.

Context:

[Source 1]
Document: refund_policy.md
Content: Customers can request a refund within 30 days of purchase.

[Source 2]
Document: refund_process.md
Content: Approved refunds are normally processed within 7 business days.

[Source 3]
Document: refund_method.md
Content: Refunds are returned to the original payment method used for the purchase.


Question:
How long do customers have to request a refund?

Answer:


This gives the generation model a clear contract:

```text
Use evidence
     ↓
Answer question
     ↓
Don't invent unsupported information
     ↓
Abstain when evidence is insufficient
```

# 6. Connecting an LLM

The exact generation model is a deployment decision.

For a tutorial, we can use an API-based model or a locally hosted model.

The important thing is the interface:

```python
answer = generate(prompt)
```

The RAG pipeline should keep retrieval and generation logically separate.

For example:

```text
retrieved_context = retrieve(...)
prompt = build_grounded_prompt(...)
answer = generate(prompt)
```

This makes each stage easier to evaluate independently.

# 7. Groundedness Is More Than Prompting

A common beginner mistake is to think:

> "I'll just write a better prompt."

Prompting is useful, but grounding is a system-level problem.

Consider:

```text
Poor retrieval
      ↓
Wrong context
      ↓
Good grounding prompt
      ↓
LLM
      ↓
Still likely a poor answer
```

The generation prompt cannot manufacture evidence that retrieval failed to provide.

Likewise:

```text
Good retrieval
      ↓
Good context
      ↓
Poor generation behavior
      ↓
Unsupported answer
```

So we need to evaluate both sides.

# 8. The Evidence Contract

A useful way to design grounded generation is to establish an explicit evidence contract.

For example:

```text
The answer may contain claims supported by the context.

If a required claim is not supported:
    do not invent it.

If the evidence is insufficient:
    abstain or explain what information is missing.
```

This changes the model's objective from:

```text
Answer every question
```

to:

```text
Answer when supported.
Abstain when unsupported.
```

That is an important property for knowledge-grounded systems.

# 9. Abstention

Abstention means the system refuses to provide an unsupported answer.

Suppose the user asks:

```text
Are all refund requests automatically approved?
```

Our context does not say.

A grounded system should be allowed to respond:

```text
The provided documents do not specify whether all refund
requests are automatically approved.
```

This may feel less impressive than producing an answer.

But in many real applications:

> **Knowing when you do not have enough evidence is more valuable than confidently guessing.**

# 10. Why Citation Matters

A useful RAG answer should often tell the user where the information came from.

Instead of:

```text
Customers can request a refund within 30 days.
```

we can produce:

```text
Customers can request a refund within 30 days of purchase.
[refund_policy.md]
```

This provides a path back to the evidence.

Citations can also make debugging easier:

```text
Answer
  ↓
Claim
  ↓
Source
  ↓
Retrieved chunk
```

This creates traceability between generation and retrieval.

# 11. Preserve Provenance

This is why the metadata we discussed in context engineering matters.

A retrieved result should ideally preserve:

```text
document_id
source
page
section
date
chunk_id
text
```

Then the generation stage can reference that provenance.

For example:

```python
{
    "text": "...",
    "source": "refund_policy.pdf",
    "page": 4,
    "section": "Refunds",
}
```

The principle is:

> **Do not throw away provenance before generation.**

If source information disappears before the LLM receives the context, producing reliable citations becomes harder.

# 12. Claim-Level Grounding

For more demanding systems, it is useful to think at the level of individual claims.

Suppose an answer contains:

```text
1. Refunds can be requested within 30 days.
2. Refunds are processed within 7 business days.
3. Refunds are always approved.
```

Our context supports:

```text
Claim 1 ✓
Claim 2 ✓
Claim 3 ✗
```

A grounded generation system should avoid claim 3.

This leads to an important evaluation question:

> **Can each important claim in the answer be traced back to evidence?**

That is a much stronger standard than simply asking whether the answer "looks good." 

# 13. Structured Generation

For some applications, free-form text is not enough.

We may want:

```text
answer
sources
confidence
abstained
```

or:

```json
{
  "answer": "...",
  "sources": ["refund_policy.md"],
  "abstained": false
}
```

Structured output can make downstream processing easier.

It can also make evaluation easier because the system has explicit fields for evidence and status.

However, structured output does not automatically make an answer truthful.

A model can still put unsupported content inside a JSON field.

The structure helps the system; it does not replace grounding.

# 14. A Simple Answer Schema

We can define the structure we want our generation stage to produce.

In [4]:
grounded_answer_schema = {
    "answer": "The answer supported by the retrieved context.",
    "sources": [
        "source document names supporting the answer"
    ],
    "abstained": False,
}

grounded_answer_schema

{'answer': 'The answer supported by the retrieved context.',
 'sources': ['source document names supporting the answer'],
 'abstained': False}

In a production application, the exact schema should match the product requirements.

For example, a legal application might need:

```text
answer
citations
document_id
page
section
```

A customer-support application might need:

```text
answer
sources
escalate
```

The schema should reflect the workflow.

# 15. Context Does Not Mean the Model Must Repeat Everything

A common mistake is to ask the LLM to summarize every retrieved chunk.

Instead, the model should answer the user's actual question.

Suppose the context contains:

```text
Refund deadline
Refund processing time
Refund payment method
Shipping policy
```

The question is:

```text
How long do I have to request a refund?
```

The answer should focus on:

```text
30 days
```

rather than dumping all retrieved information.

Good context engineering gives the model useful evidence.

Good generation uses that evidence selectively.

# 16. Handling Conflicting Evidence

Suppose the context contains:

```text
Current policy:
30 days

Archived policy:
14 days
```

A naive model might combine them incorrectly.

The generation system should have access to the provenance and metadata necessary to distinguish the sources.

If the application has an explicit rule:

```text
Current policy > archived policy
```

that rule can be incorporated into the pipeline.

If the system cannot determine which source is authoritative, it should not silently invent a resolution.

A safe response may acknowledge the conflict and identify the sources.

# 17. Grounding and Hallucination

Hallucination is often discussed as if it were solely a property of the LLM.

In RAG, it is useful to separate several failure modes:

```text
Retrieval failure
      ↓
Wrong evidence

Context failure
      ↓
Right evidence not included properly

Generation failure
      ↓
Unsupported claim despite available evidence
```

Therefore, "the model hallucinated" is often too vague to be useful.

A better debugging question is:

> **At which stage did the system lose grounding?**

# 18. A Grounded Generation Checklist

Before shipping a RAG generation stage, ask:

### Evidence
- Was relevant evidence retrieved?
- Was it preserved in the final context?

### Provenance
- Can we identify where each piece of evidence came from?

### Generation
- Does the model have explicit grounding instructions?
- Can it abstain?

### Citations
- Can important claims be traced to sources?

### Conflicts
- What happens when sources disagree?

### Evaluation
- Are unsupported claims measured?

This checklist turns grounding into an engineering problem rather than a prompt-writing exercise.

# 19. The Complete RAG Architecture

We can now see the complete pipeline we've built so far:

```text
                         Documents
                             ↓
                         Ingestion
                             ↓
                          Chunking
                             ↓
                         Embeddings
                             ↓
                       Vector Index
                             ↓
                           Query
                             ↓
                Dense + Lexical Retrieval
                             ↓
                            RRF
                             ↓
                         Reranking
                             ↓
                   Context Engineering
                             ↓
                   Grounded Generation
                             ↓
                          Answer
                         ↙     ↘
                   Sources     User
```

Every stage has a different responsibility.

That separation will become especially important when we start evaluating the system.

# 20. Debugging Generation Failures

Suppose the user receives an incorrect answer.

Don't immediately change the generation prompt.

Trace the evidence chain:

```text
1. Was the correct document retrieved?
              ↓
2. Was it ranked highly?
              ↓
3. Was it included in context?
              ↓
4. Was its provenance preserved?
              ↓
5. Did the model use the evidence?
              ↓
6. Did the answer introduce unsupported claims?
```

This gives us a systematic debugging process.

A production RAG system should make these intermediate artifacts inspectable.

# Key Takeaways

1. Retrieval quality does not guarantee grounded generation.
2. A grounded answer should be supported by the available evidence.
3. Generation prompts can establish grounding rules, but prompting alone is not enough.
4. A strong system should be able to abstain when evidence is insufficient.
5. Citations provide traceability from answer → source → retrieved evidence.
6. Preserve document provenance throughout the pipeline.
7. Think about grounding at the level of individual claims when reliability matters.
8. Structured output can make RAG systems easier to integrate and evaluate.
9. Conflicting evidence requires explicit source-authority rules or transparent handling.
10. Generation failures should be traced back through retrieval, ranking, context, and generation.
11. The objective is not to make the model answer every question.
12. The objective is to make the model answer **when the evidence supports an answer**.

The mental model is:

```text
Retrieve evidence
       ↓
Select evidence
       ↓
Preserve provenance
       ↓
Generate from evidence
       ↓
Cite evidence
       ↓
Abstain when evidence is insufficient
```

# What's Next?

We now have the complete conceptual RAG loop:

```text
Documents
   ↓
Retrieval
   ↓
Reranking
   ↓
Context
   ↓
Grounded Generation
   ↓
Answer
```

But there is a major unanswered question:

> **How do we know whether the system actually works?**

A RAG application can look impressive in a few manual tests while failing badly across a real evaluation set.

The next module focuses on **RAG Evaluation**.

We'll move from:

> "This answer looks good."

to:

> **"We can measure retrieval quality, context quality, and answer quality."**

That is where RAG engineering becomes an empirical discipline.